In [ ]:
!pip install cv2

ERROR: Could not find a version that satisfies the requirement cv2 (from versions: none)
ERROR: No matching distribution found for cv2


In [1]:
from ultralytics import YOLO

WARNING ⚠️ user config directory '/home/ubuntu/.config/Ultralytics' is not writable, using '/tmp/Ultralytics'. Set YOLO_CONFIG_DIR to override.
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/tmp/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [7]:
import cv2

ImportError: libGL.so.1: cannot open shared object file: No such file or directory

In [9]:
!pip uninstall opencv-python -y
!pip install opencv-python-headless

Found existing installation: opencv-python 4.13.0.92
Uninstalling opencv-python-4.13.0.92:
  Successfully uninstalled opencv-python-4.13.0.92
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 69.0 MB/s eta 0:00:00:00:0100:01


In [10]:
import numpy as np

def parse_polygon(coords_flat):
    """Liste [x0,y0,x1,y1,...] → array (N,2)"""
    pts = np.array(coords_flat, dtype=float).reshape(-1, 2)
    return pts

def top_mid_bottom_mid(pts):
    """
    Retourne (top_mid, bot_mid) où :
      - top_mid = milieu des points ayant y minimal (côté haut)
      - bot_mid = milieu des points ayant y maximal (côté bas)
    On sélectionne les 2 points les plus proches du min/max de y.
    """
    ys = pts[:, 1]

    # Top : 2 points avec le plus petit y
    top_idx = np.argsort(ys)[:2]
    top_mid = pts[top_idx].mean(axis=0)

    # Bottom : 2 points avec le plus grand y
    bot_idx = np.argsort(ys)[-2:]
    bot_mid = pts[bot_idx].mean(axis=0)

    return top_mid, bot_mid

def centerline_triangle(pts, half_width=0.0005):
    """
    Construit le triangle centerline :
      - sommet A : milieu du côté haut
      - sommet B : milieu côté bas décalé à gauche de half_width
      - sommet C : milieu côté bas décalé à droite de half_width
    """
    top_mid, bot_mid = top_mid_bottom_mid(pts)

    A = top_mid
    B = np.array([bot_mid[0] - half_width, bot_mid[1]])
    C = np.array([bot_mid[0] + half_width, bot_mid[1]])

    return A, B, C

def format_triangle_line(A, B, C, class_id=0):
    coords = [A[0], A[1], B[0], B[1], C[0], C[1]]
    coords_str = " ".join(f"{v:.6f}" for v in coords)
    return f"{class_id} {coords_str}"

print("Utilitaires chargés ✓")

Utilitaires chargés ✓


In [11]:
def is_centerline(coords):
    """Un triangle = exactement 3 points = 6 coordonnées."""
    return len(coords) == 6

In [12]:
def process_txt(filepath, dry_run=True):
    with open(filepath, "r") as f:
        lines = [l.strip() for l in f if l.strip()]

    new_lines = []
    centerline_line = None


    for line in lines:
      parts = line.split()
      cls = int(parts[0])
      coords = list(map(float, parts[1:]))
      if cls == 0 and is_centerline(coords):
          print(f"⏭️  Déjà traité, ignoré : {filepath}")
          return None

    for line in lines:
        parts = line.split()
        cls = int(parts[0])
        coords = list(map(float, parts[1:]))

        if cls == 0:
            # C'est la runway → calculer la centerline
            pts = parse_polygon(coords)
            A, B, C = centerline_triangle(pts, half_width=0.005)
            centerline_line = format_triangle_line(A, B, C, class_id=0)
            # Runway elle-même → reroulée en classe 1
            new_lines.append(f"1 " + " ".join(f"{v:.6f}" for v in coords))
        else:
            # Autres classes décalées de +1
            new_lines.append(f"{cls + 1} " + " ".join(f"{v:.6f}" for v in coords))

    # Centerline en tête
    if centerline_line:
        output = [centerline_line] + new_lines
    else:
        output = new_lines  # pas de runway trouvée, on laisse tel quel

    if dry_run:
        print(f"\n=== {filepath} ===")
        for l in output:
            print(l)
    else:
        with open(filepath, "w") as f:
            f.write("\n".join(output) + "\n")

    return output

In [15]:
import os
from pathlib import Path

LABELS_ROOT = "/home/ubuntu/projects/Capstone/Data/labels_rename"  # contient train/, val/, test/ etc.

txt_files = list(Path(LABELS_ROOT).rglob("*.txt"))
print(f"{len(txt_files)} fichiers .txt trouvés")

no_runway = []


for fp in txt_files:
    result = process_txt(str(fp), dry_run=False)
    # Vérifier si aucune runway n'a été trouvée
    if result is None:
        continue
    if not any(l.startswith("0 ") for l in result):
        no_runway.append(str(fp))

print(f"\n✅ Traitement terminé.")
if no_runway:
    print(f"⚠️  {len(no_runway)} fichiers sans runway (classe 0) détectée :")
    for f in no_runway:
        print(" ", f)

12261 fichiers .txt trouvés
⏭️  Déjà traité, ignoré : /home/ubuntu/projects/Capstone/Data/labels_rename/01521.txt

✅ Traitement terminé.


In [14]:
!pwd

/home/ubuntu/projects/Capstone


In [16]:
from pathlib import Path
import random
import shutil
from math import floor

ROOT = Path("/home/ubuntu/projects/Capstone/Data")
IMAGES_SRC = ROOT / "images_rename"
LABELS_SRC = ROOT / "labels_rename"
IMAGES_DST_ROOT = ROOT / "images"          # images/train, images/val, images/test
LABELS_DST_ROOT = ROOT / "labels_c_r"      # labels_c_r/train, labels_c_r/val, labels_c_r/test

SEED = 42
MOVE_FILES = True  # True = déplace, False = copie

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

assert IMAGES_SRC.exists(), f"Dossier introuvable: {IMAGES_SRC}"
assert LABELS_SRC.exists(), f"Dossier introuvable: {LABELS_SRC}"

# Récupère toutes les images
images = [p for p in IMAGES_SRC.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTS]
images.sort()

rng = random.Random(SEED)
rng.shuffle(images)

n = len(images)
n_train = floor(0.80 * n)
n_val = floor(0.10 * n)
n_test = n - n_train - n_val

splits = {
    "train": images[:n_train],
    "val": images[n_train:n_train + n_val],
    "test": images[n_train + n_val:],
}

for split in ("train", "val", "test"):
    (IMAGES_DST_ROOT / split).mkdir(parents=True, exist_ok=True)
    (LABELS_DST_ROOT / split).mkdir(parents=True, exist_ok=True)

moved = {"train": 0, "val": 0, "test": 0}
missing_label = []
dest_exists = []

op = shutil.move if MOVE_FILES else shutil.copy2

for split, imgs in splits.items():
    for img_path in imgs:
        label_path = LABELS_SRC / f"{img_path.stem}.txt"
        if not label_path.exists():
            missing_label.append(img_path.name)
            continue

        img_dst = IMAGES_DST_ROOT / split / img_path.name
        label_dst = LABELS_DST_ROOT / split / label_path.name

        if img_dst.exists() or label_dst.exists():
            dest_exists.append(img_path.name)
            continue

        op(str(img_path), str(img_dst))
        op(str(label_path), str(label_dst))
        moved[split] += 1

print("\n✅ Split terminé")
print(f"Images trouvées: {n}")
print(f"Répartition cible: train={n_train}, val={n_val}, test={n_test}")
print("Déplacées/copées:", moved)

if missing_label:
    print(f"\n⚠️  Labels manquants pour {len(missing_label)} image(s) (exemples):")
    for name in missing_label[:20]:
        print(" -", name)
    if len(missing_label) > 20:
        print(" - ...")

if dest_exists:
    print(f"\n⚠️  Déjà présent en destination pour {len(dest_exists)} image(s) (exemples):")
    for name in dest_exists[:20]:
        print(" -", name)
    if len(dest_exists) > 20:
        print(" - ...")



✅ Split terminé
Images trouvées: 12254
Répartition cible: train=9803, val=1225, test=1226
Déplacées/copées: {'train': 9790, 'val': 1223, 'test': 1226}

⚠️  Labels manquants pour 15 image(s) (exemples):
 - 11518(1).jpg
 - 11521(1).jpg
 - 11517(1).jpg
 - 11520(1).jpg
 - 11516(1).jpg
 - 11510(1).jpg
 - 11525(1).jpg
 - 11514(1).jpg
 - 11523(1).jpg
 - 11512(1).jpg
 - 11519(1).jpg
 - 11513(1).jpg
 - 11511(1).jpg
 - 11522(1).jpg
 - 11524(1).jpg


In [18]:
from pathlib import Path

SRC = Path("/home/ubuntu/projects/Capstone/Data/labels")
DST = Path("/home/ubuntu/projects/Capstone/Data/labels_2")
SPLITS = ("train", "val", "test")

assert SRC.exists(), f"Dossier introuvable: {SRC}"

DST.mkdir(parents=True, exist_ok=True)

kept_lines = 0
removed_lines = 0
files_total = 0
files_empty = 0

for split in SPLITS:
    src_split = SRC / split
    dst_split = DST / split
    dst_split.mkdir(parents=True, exist_ok=True)

    if not src_split.exists():
        print(f"⚠️  Split manquant, ignoré: {src_split}")
        continue

    for src_txt in src_split.rglob("*.txt"):
        rel = src_txt.relative_to(src_split)
        dst_txt = dst_split / rel
        dst_txt.parent.mkdir(parents=True, exist_ok=True)

        out = []
        with src_txt.open("r", encoding="utf-8") as f:
            for raw in f:
                line = raw.strip()
                if not line:
                    continue
                parts = line.split()
                try:
                    cls = int(parts[0])
                except ValueError:
                    continue

                if cls == 1:
                    out.append("0 " + " ".join(parts[1:]))
                    kept_lines += 1
                else:
                    removed_lines += 1

        dst_txt.write_text(("\n".join(out) + ("\n" if out else "")), encoding="utf-8")
        files_total += 1
        if not out:
            files_empty += 1

print("\n✅ labels_2 créé")
print(f"Fichiers traités: {files_total}")
print(f"Lignes gardées (ancienne classe 1 → nouvelle classe 0): {kept_lines}")
print(f"Lignes supprimées (toutes classes ≠ 1, dont classe 0): {removed_lines}")
print(f"Fichiers vides après filtrage: {files_empty}")
print(f"Destination: {DST}")



✅ labels_2 créé
Fichiers traités: 22493
Lignes gardées (ancienne classe 1 → nouvelle classe 0): 25522
Lignes supprimées (toutes classes ≠ 1, dont classe 0): 22497
Fichiers vides après filtrage: 2
Destination: /home/ubuntu/projects/Capstone/Data/labels_2


In [17]:
!pwd

/home/ubuntu/projects/Capstone


In [3]:
model = YOLO("yolo26n-seg.pt")  # Ultralytics va télécharger le poids si besoin

model.train(
    data="/home/ubuntu/projects/Capstone/bars_yolo_seg.yaml",
    epochs=120,       # ou 5 pour un test rapide
    imgsz=640,       # ou 1024 si le GPU le permet
    batch=16,        # ajuste si OOM
    device=0,        # GPU Colab
    workers=4,       # 2–4 en général OK
    save_period=10
)

Ultralytics 8.4.37 🚀 Python-3.12.3 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/ubuntu/projects/Capstone/bars_yolo_seg.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=120, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mas

KeyboardInterrupt: 

In [2]:
!nvidia-smi
!python -c "import torch; print(torch.cuda.is_available(), torch.version.cuda, torch.cuda.device_count())"

Thu Apr 16 02:58:39 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.126.09             Driver Version: 580.126.09     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:1E.0 Off |                    0 |
| N/A   42C    P0             27W /   70W |       0MiB /  15360MiB |     10%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
from ultralytics.utils.plotting import plot_results
plot_results('runs/segment/train/results.csv')